In [3]:
import requests
from bs4 import BeautifulSoup
import re
import pandas as pd
from datetime import datetime, timedelta

def parse_interfax_date(url):
    """Парсит новости за конкретную дату"""
    try:
        response = requests.get(url)
        soup = BeautifulSoup(response.text, 'html.parser')
        
        news_block = soup.find('div', class_='an')
        
        if not news_block:
            return []
        
        full_text = news_block.get_text()
        time_pattern = r'\d{2}:\d{2}'
        all_times = re.findall(time_pattern, full_text)
        links = news_block.find_all('a', href=True)
        
        # Извлекаем дату из URL
        date_match = re.search(r'/(\d{4})/(\d{2})/(\d{2})', url)
        if date_match:
            year, month, day = date_match.groups()
            base_date = f"{year}-{month}-{day}"
        else:
            return []
        
        data = []
        time_index = 0
        
        for link in links:
            text = link.get_text(strip=True)
            
            if text and len(text) > 10:
                time_str = all_times[time_index] if time_index < len(all_times) else "00:00"
                time_index += 1
                
                datetime_str = f"{base_date} {time_str}:00"
                
                data.append({
                    'datetime': datetime_str,
                    'text': text
                })
        
        return data
        
    except Exception as e:
        print(f"Ошибка при парсинге {url}: {e}")
        return []

def parse_interfax_period(start_year = 2022, end_year = 2025):
    """Парсит новости за период с start_year по end_year"""
    all_news = []
    
    # Генерируем даты с 2022-01-01 по 2025-12-31
    start_date = datetime(start_year, 1, 1)
    end_date = datetime(end_year, 10, 19)
    
    current_date = start_date
    total_days = (end_date - start_date).days + 1
    
    print(f"Начинаем парсинг с {start_date.date()} по {end_date.date()}")
    print(f"Всего дней: {total_days}")
    
    for day_num in range(total_days):
        date_str = current_date.strftime("%Y/%m/%d")
        url = f"https://www.interfax.ru/business/news/{date_str}"
        
        # Парсим новости за день
        daily_news = parse_interfax_date(url)
        all_news.extend(daily_news)
        
        if daily_news:
            print(f"{current_date.date()}: {len(daily_news)} новостей")
        else:
            print(f"{current_date.date()}: 0 новостей")
        
        # Переходим к следующему дню
        current_date += timedelta(days=1)
        
        # Пауза между запросами
        import time
        time.sleep(1)
    
    # Создаем итоговый DataFrame
    df = pd.DataFrame(all_news)
    
    if not df.empty:
        df = df.sort_values('datetime').reset_index(drop=True)
    
    return df

# Запускаем парсинг за весь период
print("=== ПАРСИНГ INTERFAX ЗА 2022-2025 ===")
final_df = parse_interfax_period(2022, 2025)

print(f"\n=== РЕЗУЛЬТАТ ===")
print(f"Итоговый размер датафрейма: {len(final_df)} новостей")

if not final_df.empty:
    print("\nПервые 5 записей:")
    print(final_df.head())
    print(f"\nДиапазон дат: {final_df['datetime'].min()} - {final_df['datetime'].max()}")
else:
    print("Не удалось спарсить ни одной новости!")

=== ПАРСИНГ INTERFAX ЗА 2022-2025 ===
Начинаем парсинг с 2022-01-01 по 2025-10-19
Всего дней: 1388
2022-01-01: 0 новостей
2022-01-02: 6 новостей
2022-01-03: 18 новостей
2022-01-04: 18 новостей
2022-01-05: 9 новостей
2022-01-06: 18 новостей
2022-01-07: 7 новостей
2022-01-08: 0 новостей
2022-01-09: 1 новостей
2022-01-10: 38 новостей
2022-01-11: 46 новостей
2022-01-12: 41 новостей
2022-01-13: 39 новостей
2022-01-14: 56 новостей
2022-01-15: 2 новостей
2022-01-16: 0 новостей
2022-01-17: 35 новостей
2022-01-18: 49 новостей
2022-01-19: 47 новостей
2022-01-20: 56 новостей
2022-01-21: 42 новостей
2022-01-22: 1 новостей
2022-01-23: 1 новостей
2022-01-24: 48 новостей
2022-01-25: 52 новостей
2022-01-26: 52 новостей
2022-01-27: 55 новостей
2022-01-28: 42 новостей
2022-01-29: 1 новостей
2022-01-30: 2 новостей
2022-01-31: 46 новостей
2022-02-01: 41 новостей
2022-02-02: 63 новостей
2022-02-03: 53 новостей
2022-02-04: 50 новостей
2022-02-05: 2 новостей
2022-02-06: 1 новостей
2022-02-07: 52 новостей
202

2022-12-08: 68 новостей
2022-12-09: 73 новостей
2022-12-10: 3 новостей
2022-12-11: 3 новостей
2022-12-12: 68 новостей
2022-12-13: 63 новостей
2022-12-14: 72 новостей
2022-12-15: 75 новостей
2022-12-16: 89 новостей
2022-12-17: 3 новостей
2022-12-18: 0 новостей
2022-12-19: 58 новостей
2022-12-20: 67 новостей
2022-12-21: 72 новостей
2022-12-22: 76 новостей
2022-12-23: 64 новостей
2022-12-24: 3 новостей
2022-12-25: 2 новостей
2022-12-26: 52 новостей
2022-12-27: 80 новостей
2022-12-28: 70 новостей
2022-12-29: 60 новостей
2022-12-30: 47 новостей
2022-12-31: 5 новостей
2023-01-01: 9 новостей
2023-01-02: 2 новостей
2023-01-03: 20 новостей
2023-01-04: 18 новостей
2023-01-05: 17 новостей
2023-01-06: 18 новостей
2023-01-07: 2 новостей
2023-01-08: 2 новостей
2023-01-09: 50 новостей
2023-01-10: 62 новостей
2023-01-11: 60 новостей
2023-01-12: 60 новостей
2023-01-13: 76 новостей
2023-01-14: 4 новостей
2023-01-15: 4 новостей
2023-01-16: 60 новостей
2023-01-17: 64 новостей
2023-01-18: 72 новостей
2023-

2023-11-19: 0 новостей
2023-11-20: 61 новостей
2023-11-21: 58 новостей
2023-11-22: 68 новостей
2023-11-23: 71 новостей
2023-11-24: 68 новостей
2023-11-25: 6 новостей
2023-11-26: 1 новостей
2023-11-27: 59 новостей
2023-11-28: 63 новостей
2023-11-29: 100 новостей
2023-11-30: 114 новостей
2023-12-01: 69 новостей
2023-12-02: 1 новостей
2023-12-03: 0 новостей
2023-12-04: 74 новостей
2023-12-05: 82 новостей
2023-12-06: 92 новостей
2023-12-07: 102 новостей
2023-12-08: 83 новостей
2023-12-09: 1 новостей
2023-12-10: 5 новостей
2023-12-11: 60 новостей
2023-12-12: 73 новостей
2023-12-13: 70 новостей
2023-12-14: 76 новостей
2023-12-15: 82 новостей
2023-12-16: 0 новостей
2023-12-17: 8 новостей
2023-12-18: 89 новостей
2023-12-19: 67 новостей
2023-12-20: 100 новостей
2023-12-21: 67 новостей
2023-12-22: 64 новостей
2023-12-23: 4 новостей
2023-12-24: 2 новостей
2023-12-25: 58 новостей
2023-12-26: 58 новостей
2023-12-27: 86 новостей
2023-12-28: 78 новостей
2023-12-29: 62 новостей
2023-12-30: 3 новостей


2024-10-15: 74 новостей
2024-10-16: 86 новостей
2024-10-17: 95 новостей
2024-10-18: 73 новостей
2024-10-19: 2 новостей
2024-10-20: 6 новостей
2024-10-21: 57 новостей
2024-10-22: 93 новостей
2024-10-23: 85 новостей
2024-10-24: 69 новостей
2024-10-25: 82 новостей
2024-10-26: 3 новостей
2024-10-27: 0 новостей
2024-10-28: 58 новостей
2024-10-29: 70 новостей
2024-10-30: 100 новостей
2024-10-31: 97 новостей
2024-11-01: 60 новостей
2024-11-02: 45 новостей
2024-11-03: 5 новостей
2024-11-04: 4 новостей
2024-11-05: 65 новостей
2024-11-06: 66 новостей
2024-11-07: 69 новостей
2024-11-08: 79 новостей
2024-11-09: 3 новостей
2024-11-10: 1 новостей
2024-11-11: 65 новостей
2024-11-12: 64 новостей
2024-11-13: 83 новостей
2024-11-14: 79 новостей
2024-11-15: 80 новостей
2024-11-16: 1 новостей
2024-11-17: 0 новостей
2024-11-18: 54 новостей
2024-11-19: 89 новостей
2024-11-20: 79 новостей
2024-11-21: 73 новостей
2024-11-22: 73 новостей
2024-11-23: 2 новостей
2024-11-24: 1 новостей
2024-11-25: 67 новостей
202

2025-09-25: 89 новостей
2025-09-26: 78 новостей
2025-09-27: 1 новостей
2025-09-28: 1 новостей
2025-09-29: 68 новостей
2025-09-30: 92 новостей
2025-10-01: 97 новостей
2025-10-02: 75 новостей
2025-10-03: 68 новостей
2025-10-04: 0 новостей
2025-10-05: 3 новостей
2025-10-06: 83 новостей
2025-10-07: 82 новостей
2025-10-08: 97 новостей
2025-10-09: 100 новостей
2025-10-10: 73 новостей
2025-10-11: 0 новостей
2025-10-12: 5 новостей
2025-10-13: 62 новостей
2025-10-14: 83 новостей
2025-10-15: 107 новостей
2025-10-16: 80 новостей
2025-10-17: 64 новостей
2025-10-18: 3 новостей
2025-10-19: 0 новостей

=== РЕЗУЛЬТАТ ===
Итоговый размер датафрейма: 71117 новостей

Первые 5 записей:
              datetime                                               text
0  2022-01-02 09:37:00  Экспорт нефти из РФ в дальнее зарубежье в 2021...
1  2022-01-02 09:42:00  Добыча нефти в РФ в 2021 году повысилась до 52...
2  2022-01-02 09:46:00    Добыча газа в России в 2021 году выросла на 10%
3  2022-01-02 15:18:00  Милле

In [6]:
final_df.to_csv('interfax_news_2022_2025.csv', index = False, encoding = 'utf-8')

In [7]:
df_2 = pd.read_csv('data/news_interfax.csv')
df_2.head()

,datetime,text
0,2022-01-02 09:37:00,"Экспорт нефти из РФ в дальнее зарубежье в 2021 году снизился на 2,2%"
1,2022-01-02 09:42:00,"Добыча нефти в РФ в 2021 году повысилась до 524,05 млн тонн"
2,2022-01-02 09:46:00,Добыча газа в России в 2021 году выросла на 10%
3,2022-01-02 15:18:00,"Миллер заявил, что в ""Газпроме"" ждут за 2021 год максимальную за его историю прибыль"
4,2022-01-02 15:21:00,"""Газпром"" в 2021 году добыл 514,8 млрд куб. м газа"


In [45]:
final_df.shape

(71117, 4)

In [10]:
df_2[df_2['datetime'] >= '2022-04-01']['text']

4427              Фондовые индексы США закрылись снижением, показав максимальное квартальное падение за 2 года
4428                              Цены на нефть продолжили падать после заявления Байдена по нефтяным резервам
4429                                  Европейские покупатели сохраняют высокие заявки на отбор газа "Газпрома"
4430                                            ЦБ смягчит ограничения на переводы средств за рубеж для физлиц
4431                                       Индекс PMI обрабатывающих отраслей РФ в марте рухнул до 44,1 пункта
                                                         ...                                                  
71112                                       Индекс Мосбиржи превысил 2720 пунктов на геополитическом оптимизме
71113                         Рубль в пятницу сильно упал в паре с юанем, растеряв весь рост предыдущих сессий
71114                                                  "Газпром" подал к Linde иск в суд РФ о возмещении вреда
7

In [11]:
df.head()

,datetime,title,full_text,url
0,2022-04-01 08:36:00,"Фондовые индексы США закрылись снижением, показав максимальное квартальное падение за 2 года","Фондовые индексы США закрылись снижением, показав максимальное квартальное падение за 2 года\nМосква. 1 апреля. INTERFAX.RU - Американские фондовые индексы завершили снижением вторые торги подряд в четверг, закрывшись на сессионных минимумах.\nПри этом все три индикатора продемонстрировали самый существенный квартальный спад за два года,\nпишет\nMarketWatch. В то же время по итогам марта индексы выросли.\n""Худший квартал за два года не так плох, поскольку индекс S&P 500 находится примерно в 5% от рекордных максимумов"", - отметил старший рыночный аналитик Oanda Эдвард Мойя.\nТрейдеры оценивали новую порцию статистических данных и следили за новостями, касающимися ситуации на Украине.\nКоличество американцев, впервые обратившихся за пособием по безработице, на прошлой неделе увеличилось на 14 тыс. - до 202 тыс. человек, сообщается в отчете министерства труда США.\nСогласно уточненным данным, неделей ранее число обращений составило 188 тыс., а не 187 тыс., как сообщалось ранее.\nОпрошенные Bloomberg аналитики в среднем ожидали повышения числа заявок до 196 тыс. с ранее объявленного уровня. Респонденты Trading Economics прогнозировали 197 тыс.\nТем временем расходы населения США в феврале выросли на 0,2% по сравнению с предыдущим месяцем, свидетельствуют данные министерства торговли страны.\nДоходы американцев увеличились на 0,5%.\nЭксперты, опрошенные агентством Bloomberg, прогнозировали подъем обоих показателей на 0,5%.\nАкции ряда технологических компаний подешевели по итогам торгов в четверг. Так, бумаги Advanced Micro Devices Inc. потеряли 8,3% после того, как аналитики Barclays ухудшили рекомендацию по акциям компании.\nЦена бумаг HP Inc. уменьшилась на 6,54%, Dell Technologies Inc. - на 7,6% на фоне ухудшения рекомендаций аналитиками Morgan Stanley.\nАкции Wendy's Co. подешевели на 2,44%. Третья по величине сеть ресторанов быстрого питания в США откроет первый виртуальный ресторан в метавселенной, разрабатываемой компанией Meta Platforms Inc. (признана в РФ экстремистской организацией и запрещена).\nБумаги Intel Corp. потеряли в цене 3,64%. Компания покупает израильского разработчика и поставщика программного обеспечения для оптимизации Granulate Cloud Solutions Ltd., при этом сумма сделки не раскрывается.",https://www.interfax.ru/world/832542
1,2022-04-01 09:07:00,Цены на нефть продолжили падать после заявления Байдена по нефтяным резервам,"Цены на нефть продолжили падать после заявления Байдена по нефтяным резервам\nМосква. 1 апреля. INTERFAX.RU - Цены на нефть продолжают снижаться в пятницу после падения накануне на заявлении президента США Джо Байдена о намерении направлять на рынок в среднем по 1 млн баррелей нефти в сутки\nиз стратегического резерва\n(SPR) на протяжении следующих шести месяцев.\nБайден, выступая в Белом доме, призвал американские компании наращивать добычу, чтобы добиться снижения цен на нефть. Это, среди прочего, позволит снизить роль экспорта энергоносителей из РФ, заявил он.\nКроме того, Байден предложил рассмотреть в Конгрессе меры, которые подтолкнули бы компании в США к более активной добыче нефти.\nСтоимость июньских фьючерсов на нефть Brent на лондонской бирже ICE Futures к 8:10 по московскому времени в пятницу составила $104,35 за баррель, что на $0,36 (0,34%) ниже цены на закрытие предыдущей сессии. По итогам торгов в четверг эти контракты подешевели на $6,73 (6%), до $104,71 за баррель.\nЦена фьючерсов на нефть WTI на май на электронных торгах Нью-йоркской товарной биржи (NYMEX) составили к этому времени $99,66 за баррель, что на $0,62 (0,62%) ниже итогового значения предыдущей сессии. В четверг стоимость этих контрактов упала на $7,54 (7%), до $100,28 за баррель.\nВ марте Brent подорожала на 6,9%, WTI - на 4,8%, за первый квартал цены выросли соответственно на 39% и 33%.\n""В принципе, это временная мера, призванная минимизировать весенний по

In [1]:
import requests
from bs4 import BeautifulSoup
import re
import pandas as pd
from datetime import datetime, timedelta
import time

def parse_news_article_safe(url):
    """Парсит полный текст новости, используя те же настройки, что и для заголовков"""
    try:
        response = requests.get(url)
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # Ищем текст новости в разных возможных контейнерах
        article_elem = (soup.find('article') or 
                       soup.find('div', {'itemprop': 'articleBody'}) or 
                       soup.find('div', class_='text') or
                       soup.find('div', class_='at'))
        
        if article_elem:
            # Очищаем текст от ненужных элементов
            for elem in article_elem.find_all(['script', 'style', 'iframe', 'div', 'span']):
                elem.decompose()
            
            full_text = article_elem.get_text('\n', strip=True)
            return full_text if full_text else ""
        else:
            return ""
            
    except Exception as e:
        print(f"    Ошибка при парсинге статьи: {e}")
        return ""

def parse_interfax_date(url):
    """Парсит новости за конкретную дату"""
    try:
        response = requests.get(url)
        soup = BeautifulSoup(response.text, 'html.parser')
        
        news_block = soup.find('div', class_='an')
        
        if not news_block:
            return []
        
        full_text = news_block.get_text()
        time_pattern = r'\d{2}:\d{2}'
        all_times = re.findall(time_pattern, full_text)
        links = news_block.find_all('a', href=True)
        
        # Извлекаем дату из URL
        date_match = re.search(r'/(\d{4})/(\d{2})/(\d{2})', url)
        if date_match:
            year, month, day = date_match.groups()
            base_date = f"{year}-{month}-{day}"
        else:
            return []
        
        data = []
        time_index = 0
        
        for i, link in enumerate(links):
            text = link.get_text(strip=True)
            href = link.get('href')
            
            if text and len(text) > 10:
                time_str = all_times[time_index] if time_index < len(all_times) else "00:00"
                time_index += 1
                
                datetime_str = f"{base_date} {time_str}:00"
                full_url = f"https://www.interfax.ru{href}" if href.startswith('/') else href
                
                # Парсим полный текст новости
                if i < 3:  # Показываем прогресс для первых 3 новостей
                    print(f"    Парсим новость {i+1}: {text[:50]}...")
                
                article_text = parse_news_article_safe(full_url)
                
                data.append({
                    'datetime': datetime_str,
                    'title': text,  # заголовок
                    'full_text': article_text,  # полный текст
                    'url': full_url
                })
                
                # Короткая пауза между запросами
                time.sleep(0.01)
        
        return data
        
    except Exception as e:
        print(f"Ошибка при парсинге {url}: {e}")
        return []

def parse_interfax_period(start_year=2022, end_year=2025):
    """Парсит новости за период с start_year по end_year"""
    all_news = []
    
    # Генерируем даты с 2022-01-01 по 2025-12-31
    start_date = datetime(start_year, 4, 1)
    end_date = datetime(end_year, 10, 19)
    
    current_date = start_date
    total_days = (end_date - start_date).days + 1
    
    print(f"Начинаем парсинг с {start_date.date()} по {end_date.date()}")
    print(f"Всего дней: {total_days}")
    print(f"Ориентировочное время: ~{total_days * 2} минут")
    print("=" * 50)
    
    for day_num in range(total_days):
        date_str = current_date.strftime("%Y/%m/%d")
        url = f"https://www.interfax.ru/business/news/{date_str}"
        
        print(f"[{day_num+1}/{total_days}] {current_date.date()}: ", end="", flush=True)
        
        # Парсим новости за день
        daily_news = parse_interfax_date(url)
        
        if daily_news:
            all_news.extend(daily_news)
            print(f"{len(daily_news)} новостей (всего: {len(all_news)})")
        else:
            print("0 новостей")
        
        # Переходим к следующему дню
        current_date += timedelta(days=1)
        
        # Пауза между днями
        time.sleep(0.5)
        
        # Промежуточное сохранение каждые 30 дней
        if (day_num + 1) % 30 == 0 and all_news:
            temp_df = pd.DataFrame(all_news)
            temp_df.to_csv(f'interfax_temp_{day_num+1}.csv', index=False, encoding='utf-8')
            print(f"  -> Сохранено во временный файл")
    
    # Создаем итоговый DataFrame
    df = pd.DataFrame(all_news)
    
    if not df.empty:
        df = df.sort_values('datetime').reset_index(drop=True)
    
    return df

# Запускаем парсинг за весь период
print("=== ПАРСИНГ INTERFAX ЗА 2022-2025 ===")
final_df = parse_interfax_period(2022, 2025)

print(f"\n=== РЕЗУЛЬТАТ ===")
print(f"Итоговый размер датафрейма: {len(final_df)} новостей")

if not final_df.empty:
    print("\nПервые 3 записи:")
    for i, row in final_df.head(3).iterrows():
        print(f"\n{i+1}. {row['datetime']}")
        print(f"   Заголовок: {row['title']}")
        print(f"   Текст: {row['full_text'][:100]}...")
        print(f"   URL: {row['url']}")
    
    print(f"\nДиапазон дат: {final_df['datetime'].min()} - {final_df['datetime'].max()}")
    
    # Сохраняем результат
    final_df.to_csv('interfax_complete_2022_2025.csv', index=False, encoding='utf-8')
    print(f"\nСохранено в interfax_complete_2022_2025.csv")
    
else:
    print("Не удалось спарсить ни одной новости!")

=== ПАРСИНГ INTERFAX ЗА 2022-2025 ===
Начинаем парсинг с 2022-04-01 по 2025-10-19
Всего дней: 1298
Ориентировочное время: ~2596 минут
[1/1298] 2022-04-01:     Парсим новость 1: Shell может столкнуться с проблемами при оплате "Г...
    Парсим новость 2: Эксперт счел, что Британия не сможет ограничить де...
    Парсим новость 3: Сахарные заводы "Продимекса" начали переработку им...
88 новостей (всего: 88)
[2/1298] 2022-04-02:     Парсим новость 1: Власти Нидерландов призвали жителей уменьшить отоп...
    Парсим новость 2: Природный газ из России перестал поступать в стран...
    Парсим новость 3: Поставки газа из Азербайджана в Италию в 2022 году...
8 новостей (всего: 96)
[3/1298] 2022-04-03:     Парсим новость 1: Заявка "Газпрома" на транзит через Украину осталас...
    Парсим новость 2: Глава "Росатома" заявил, что госкорпорация не план...
    Парсим новость 3: Белоруссия и Россия создали Совет промышленников...
3 новостей (всего: 99)
[4/1298] 2022-04-04:     Парсим новость 1: Правител

    Ошибка при парсинге статьи: HTTPSConnectionPool(host='www.interfax.ru', port=443): Max retries exceeded with url: /business/838813 (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x000002176A052420>, 'Connection to www.interfax.ru timed out. (connect timeout=None)'))
117 новостей (всего: 1701)
[30/1298] 2022-04-30:     Парсим новость 1: В МИД РФ заявили, что санкции стимулируют переход ...
    Парсим новость 2: Правительство РФ смягчило ограничения на вывоз мин...
    Парсим новость 3: Европа увеличивает заявки на поставку российского ...
5 новостей (всего: 1706)
  -> Сохранено во временный файл
[31/1298] 2022-05-01:     Парсим новость 1: Вступил в силу закон, ограничивающий банки РФ в пе...
    Парсим новость 2: "Газпром" за январь-апрель нарастил поставки в Кит...
    Парсим новость 3: Приставы РФ принудительно взыщут с Google свыше 7,...
8 новостей (всего: 1714)
[32/1298] 2022-05-02:     Парсим новость 1: "Интеррос" купил процессинговую компанию Unit

87 новостей (всего: 3010)
[58/1298] 2022-05-28:     Парсим новость 1: Олег Бударгин может вернуться в совет директоров "...
    Парсим новость 2: Сбербанк снизил ставки по вкладам, потребкредитам ...
    Парсим новость 3: Утверждены параметры льготного кредитования для си...
4 новостей (всего: 3014)
[59/1298] 2022-05-29:     Парсим новость 1: На саммите ЕС обсудят возможность конфискации росс...
    Парсим новость 2: Министр ЕЭК призвал создать биржевое пространство ...
    Парсим новость 3: Евросовет попытается найти компромиссное решение п...
8 новостей (всего: 3022)
[60/1298] 2022-05-30:     Парсим новость 1: "Мегафон" решил не публиковать финансовую отчетнос...
    Парсим новость 2: Топ-менеджеры "Интерроса" и выходцы из группы ВТБ ...
    Парсим новость 3: Выдача ипотеки в России в апреле упала к марту в 3...
77 новостей (всего: 3099)
  -> Сохранено во временный файл
[61/1298] 2022-05-31:     Парсим новость 1: FT узнала о решении Британии и ЕС запретить страхо...
    Парсим новост

    Парсим новость 2: В Еврокомиссии призвали страны ЕС подготовиться к ...
    Парсим новость 3: Siemens купит разработчика ПО Brightly Software за...
81 новостей (всего: 4479)
[89/1298] 2022-06-28:     Парсим новость 1: Siemens купил долю в американской сети VW станций ...
    Парсим новость 2: Berkshire Hathaway увеличила долю в Occidental Pet...
    Парсим новость 3: Конвертация депозитарных расписок МТС в обыкновенн...
74 новостей (всего: 4553)
[90/1298] 2022-06-29:     Парсим новость 1: Минэкономразвития не поддержало законопроект о при...
    Парсим новость 2: Непротивление казне согласием. Репортаж...
    Парсим новость 3: Саид Гуцериев намерен продать свой пакет в холдинг...
86 новостей (всего: 4639)
  -> Сохранено во временный файл
[91/1298] 2022-06-30:     Парсим новость 1: Крым возобновляет автобусное сообщение с Херсонщин...
    Парсим новость 2: СПГ-проект "Сахалин-2" получит оператора в российс...
    Парсим новость 3: Старт пассажирских поездов между Крымом, Херсонщин..

85 новостей (всего: 6212)
[120/1298] 2022-07-29:     Парсим новость 1: Нефтепровод Тенгиз-Новороссийск заработал в штатно...
    Парсим новость 2: Ракета Electron 2 августа стартует на орбиту с раз...
    Парсим новость 3: Европейский рынок акций вырос в июле максимально с...
80 новостей (всего: 6292)
  -> Сохранено во временный файл
[121/1298] 2022-07-30:     Парсим новость 1: В США выразили надежду на скорое исполнение соглаш...
    Парсим новость 2: Глава ЕК сообщила о проблемах с газоснабжением из ...
    Парсим новость 3: "Газпром" прекратил поставки в Латвию в связи с на...
5 новостей (всего: 6297)
[122/1298] 2022-07-31:     Парсим новость 1: В Минсельхозе Белоруссии не намерены экспортироват...
    Парсим новость 2: Тарифная квота на экспорт за пределы ЕАЭС лома и о...
2 новостей (всего: 6299)
[123/1298] 2022-08-01:     Парсим новость 1: "КАМАЗ" и Москва договорились о паритетном участии...
    Парсим новость 2: Саудовская Аравия в июле экспортировала рекордный ...
    Парсим но

...

    Парсим новость 3: ЕК выделила Украине 1 млрд евро в счет кредитной и...
20 новостей (всего: 59640)
  -> Сохранено во временный файл
[1171/1298] 2025-06-14:     Парсим новость 1: В Казахстане заявили, что у КНР наиболее высокий у...
    Парсим новость 2: Китайская СNNC возглавит консорциум по строительст...
    Парсим новость 3: АЭС в Казахстане будет построена на основе российс...
4 новостей (всего: 59644)
[1172/1298] 2025-06-15: 0 новостей
[1173/1298] 2025-06-16:     Парсим новость 1: Реальный эффективный курс рубля в мае вырос на 2,7...
    Парсим новость 2: Рубль по итогам волатильных торгов понедельника сл...
    Парсим новость 3: Рынок акций РФ начал неделю с отката ниже 2740 пун...
38 новостей (всего: 59682)
[1174/1298] 2025-06-17:     Парсим новость 1: Yum! Brands в октябре возглавит нынешний финдирект...
    Парсим новость 2: Владельцы биржевых бондов "Домодедово" не поддержа...
    Парсим новость 3: Канада вводит санкции против более 40 юрлиц из РФ ...
57 новостей (всего: 

5 новостей (всего: 61033)
[1200/1298] 2025-07-13: 0 новостей
  -> Сохранено во временный файл
[1201/1298] 2025-07-14:     Парсим новость 1: Рубль вырос в паре с юанем на фоне отсутствия серь...
    Парсим новость 2: Объем грузоперевозок по сети РЖД снизился в I полу...
    Парсим новость 3: На новых заявлениях Трампа Brent подешевела до $69...
49 новостей (всего: 61082)
[1202/1298] 2025-07-15:     Парсим новость 1: В кабмин внесен проект поправок о налоговом вычете...
    Парсим новость 2: В Банке России оценили инфляцию в июне как близкую...
    Парсим новость 3: Рубль во вторник умеренно вырос к юаню на фоне вне...
58 новостей (всего: 61140)
[1203/1298] 2025-07-16:     Парсим новость 1: Опрос ЦБ показал, что охлаждение внутреннего спрос...
    Парсим новость 2: Цены производителей на бензин в РФ в июне выросли ...
    Парсим новость 3: Bloomberg сообщает, что Трамп может вскоре уволить...
58 новостей (всего: 61198)
[1204/1298] 2025-07-17:     Парсим новость 1: Принадлежащая менеджмен

    Парсим новость 3: Инвестиции в нефтегазовый сектор Норвегии достигну...
68 новостей (всего: 62609)
[1233/1298] 2025-08-15:     Парсим новость 1: Путин поручил зафиксировать сроки давности для осп...
    Парсим новость 2: Суд завершил конкурсное производство на "Евродоне"...
    Парсим новость 3: Правительство подготовит предложения по созданию "...
74 новостей (всего: 62683)
[1234/1298] 2025-08-16:     Парсим новость 1: Ставки пошлин на экспорт пшеницы, ячменя и кукуруз...
1 новостей (всего: 62684)
[1235/1298] 2025-08-17:     Парсим новость 1: Ростовская область снизила сбор зерновых на 24%, д...
1 новостей (всего: 62685)
[1236/1298] 2025-08-18:     Парсим новость 1: Bloomberg сообщил о намерении властей США получить...
    Парсим новость 2: Доходность 30-летних британских облигаций с защито...
    Парсим новость 3: Рубль в понедельник опустился к юаню в ожидании вн...
49 новостей (всего: 62734)
[1237/1298] 2025-08-19:     Парсим новость 1: Латвия вложит 14 млн евро в капитал airBa

[1265/1298] 2025-09-16:     Парсим новость 1: Оператор Нежданинского месторождения "Полиметалла"...
    Парсим новость 2: Эстонская Fermi Energia и канадская Aecon будут со...
    Парсим новость 3: Каллас подтвердила планы ЕК ввести ограничения в т...
58 новостей (всего: 64222)
[1266/1298] 2025-09-17:     Парсим новость 1: Пауэлл заявил, что никто не знает, в каком состоян...
    Парсим новость 2: ФРС подтвердила прогноз инфляции на 2025 г., прогн...
    Парсим новость 3: ФРС снизила базовую ставку впервые с декабря - на ...
84 новостей (всего: 64306)
[1267/1298] 2025-09-18:     Парсим новость 1: В ЦБ заявили, что идея субсидий для IPO увязла в с...
    Парсим новость 2: Новые критерии участия в "Сахалине-1" коснутся инд...
    Парсим новость 3: Казначейство планирует в ноябре начать проработку ...
84 новостей (всего: 64390)
[1268/1298] 2025-09-19:     Парсим новость 1: Гендиректором оператора российских космодромов ЦЭН...
    Парсим новость 2: Трамп обсудит с Эрдоганом торговые соглаш

[1295/1298] 2025-10-16:     Парсим новость 1: На подъем затонувшего в Мурманске плавдока ПД-50 в...
    Парсим новость 2: В Думу повторно внесли законопроект о регулировани...
    Парсим новость 3: Лагард считает, что нынешний уровень ставок поможе...
80 новостей (всего: 66062)
[1296/1298] 2025-10-17:     Парсим новость 1: Рубль в пятницу сильно упал в паре с юанем, растер...
    Парсим новость 2: Индекс Мосбиржи превысил 2720 пунктов на геополити...
    Парсим новость 3: Apple будет транслировать "Формулу-1" в США на экс...
64 новостей (всего: 66126)
[1297/1298] 2025-10-18:     Парсим новость 1: Повышение утильсбора не затронет всех импортеров, ...
    Парсим новость 2: "ВСМПО-Ависма" с 1 декабря планирует перевести адм...
    Парсим новость 3: "Газпром" подал к Linde иск в суд РФ о возмещении ...
3 новостей (всего: 66129)
[1298/1298] 2025-10-19:     Парсим новость 1: Оренбургский ГПЗ после атаки БПЛА временно прекрат...
1 новостей (всего: 66130)

=== РЕЗУЛЬТАТ ===
Итоговый размер дат

In [2]:
df = pd.read_csv('interfax_complete_2022_2025.csv')
df.head()

,datetime,title,full_text,url
0,2022-04-01 08:36:00,"Фондовые индексы США закрылись снижением, пока...","Фондовые индексы США закрылись снижением, пока...",https://www.interfax.ru/world/832542
1,2022-04-01 09:07:00,Цены на нефть продолжили падать после заявлени...,Цены на нефть продолжили падать после заявлени...,https://www.interfax.ru/business/832550
2,2022-04-01 09:13:00,Европейские покупатели сохраняют высокие заявк...,Европейские покупатели сохраняют высокие заявк...,https://www.interfax.ru/business/832553
3,2022-04-01 09:15:00,ЦБ смягчит ограничения на переводы средств за ...,ЦБ смягчит ограничения на переводы средств за ...,https://www.interfax.ru/business/832554
4,2022-04-01 09:16:00,Индекс PMI обрабатывающих отраслей РФ в марте ...,Индекс PMI обрабатывающих отраслей РФ в марте ...,https://www.interfax.ru/business/832557


In [3]:
pd.set_option('display.max_colwidth', None)
pd.set_option('display.expand_frame_repr', False)

In [4]:
df.head()

,datetime,title,full_text,url
0,2022-04-01 08:36:00,"Фондовые индексы США закрылись снижением, показав максимальное квартальное падение за 2 года","Фондовые индексы США закрылись снижением, показав максимальное квартальное падение за 2 года\nМосква. 1 апреля. INTERFAX.RU - Американские фондовые индексы завершили снижением вторые торги подряд в четверг, закрывшись на сессионных минимумах.\nПри этом все три индикатора продемонстрировали самый существенный квартальный спад за два года,\nпишет\nMarketWatch. В то же время по итогам марта индексы выросли.\n""Худший квартал за два года не так плох, поскольку индекс S&P 500 находится примерно в 5% от рекордных максимумов"", - отметил старший рыночный аналитик Oanda Эдвард Мойя.\nТрейдеры оценивали новую порцию статистических данных и следили за новостями, касающимися ситуации на Украине.\nКоличество американцев, впервые обратившихся за пособием по безработице, на прошлой неделе увеличилось на 14 тыс. - до 202 тыс. человек, сообщается в отчете министерства труда США.\nСогласно уточненным данным, неделей ранее число обращений составило 188 тыс., а не 187 тыс., как сообщалось ранее.\nОпрошенные Bloomberg аналитики в среднем ожидали повышения числа заявок до 196 тыс. с ранее объявленного уровня. Респонденты Trading Economics прогнозировали 197 тыс.\nТем временем расходы населения США в феврале выросли на 0,2% по сравнению с предыдущим месяцем, свидетельствуют данные министерства торговли страны.\nДоходы американцев увеличились на 0,5%.\nЭксперты, опрошенные агентством Bloomberg, прогнозировали подъем обоих показателей на 0,5%.\nАкции ряда технологических компаний подешевели по итогам торгов в четверг. Так, бумаги Advanced Micro Devices Inc. потеряли 8,3% после того, как аналитики Barclays ухудшили рекомендацию по акциям компании.\nЦена бумаг HP Inc. уменьшилась на 6,54%, Dell Technologies Inc. - на 7,6% на фоне ухудшения рекомендаций аналитиками Morgan Stanley.\nАкции Wendy's Co. подешевели на 2,44%. Третья по величине сеть ресторанов быстрого питания в США откроет первый виртуальный ресторан в метавселенной, разрабатываемой компанией Meta Platforms Inc. (признана в РФ экстремистской организацией и запрещена).\nБумаги Intel Corp. потеряли в цене 3,64%. Компания покупает израильского разработчика и поставщика программного обеспечения для оптимизации Granulate Cloud Solutions Ltd., при этом сумма сделки не раскрывается.",https://www.interfax.ru/world/832542
1,2022-04-01 09:07:00,Цены на нефть продолжили падать после заявления Байдена по нефтяным резервам,"Цены на нефть продолжили падать после заявления Байдена по нефтяным резервам\nМосква. 1 апреля. INTERFAX.RU - Цены на нефть продолжают снижаться в пятницу после падения накануне на заявлении президента США Джо Байдена о намерении направлять на рынок в среднем по 1 млн баррелей нефти в сутки\nиз стратегического резерва\n(SPR) на протяжении следующих шести месяцев.\nБайден, выступая в Белом доме, призвал американские компании наращивать добычу, чтобы добиться снижения цен на нефть. Это, среди прочего, позволит снизить роль экспорта энергоносителей из РФ, заявил он.\nКроме того, Байден предложил рассмотреть в Конгрессе меры, которые подтолкнули бы компании в США к более активной добыче нефти.\nСтоимость июньских фьючерсов на нефть Brent на лондонской бирже ICE Futures к 8:10 по московскому времени в пятницу составила $104,35 за баррель, что на $0,36 (0,34%) ниже цены на закрытие предыдущей сессии. По итогам торгов в четверг эти контракты подешевели на $6,73 (6%), до $104,71 за баррель.\nЦена фьючерсов на нефть WTI на май на электронных торгах Нью-йоркской товарной биржи (NYMEX) составили к этому времени $99,66 за баррель, что на $0,62 (0,62%) ниже итогового значения предыдущей сессии. В четверг стоимость этих контрактов упала на $7,54 (7%), до $100,28 за баррель.\nВ марте Brent подорожала на 6,9%, WTI - на 4,8%, за первый квартал цены выросли соответственно на 39% и 33%.\n""В принципе, это временная мера, призванная минимизировать весенний по

In [9]:
df.shape

(66130, 4)